In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import PowerTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report,confusion_matrix

In [2]:
cols = ['loan_amnt', 'term', 'int_rate', 'grade', 'emp_length', 'home_ownership', 
        'annual_inc', 'purpose', 'dti', 'loan_status','installment','verification_status']

In [3]:
data = pd.read_csv(r'C:\Users\Mayur23\Downloads\accepted_2007_to_2018Q4.csv.gz', usecols = cols)

In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 12 columns):
 #   Column               Dtype  
---  ------               -----  
 0   loan_amnt            float64
 1   term                 str    
 2   int_rate             float64
 3   installment          float64
 4   grade                str    
 5   emp_length           str    
 6   home_ownership       str    
 7   annual_inc           float64
 8   verification_status  str    
 9   loan_status          str    
 10  purpose              str    
 11  dti                  float64
dtypes: float64(5), str(7)
memory usage: 207.0 MB


In [5]:
data.describe()

,loan_amnt,int_rate,installment,annual_inc,dti
count,2.260668e+06,2.260668e+06,2.260668e+06,2.260664e+06,2.258957e+06
mean,1.504693e+04,1.309283e+01,4.458068e+02,7.799243e+04,1.882420e+01
std,9.190245e+03,4.832138e+00,2.671735e+02,1.126962e+05,1.418333e+01
min,5.000000e+02,5.310000e+00,4.930000e+00,0.000000e+00,-1.000000e+00
25%,8.000000e+03,9.490000e+00,2.516500e+02,4.600000e+04,1.189000e+01
50%,1.290000e+04,1.262000e+01,3.779900e+02,6.500000e+04,1.784000e+01
75%,2.000000e+04,1.599000e+01,5.933200e+02,9.300000e+04,2.449000e+01
max,4.000000e+04,3.099000e+01,1.719830e+03,1.100000e+08,9.990000e+02


In [6]:
data['term'].unique()

<StringArray>
[' 36 months', ' 60 months', nan]
Length: 3, dtype: str

In [7]:
map_term = {
    ' 36 months' : 'Short-term',
    ' 60 months' : 'Long-term'
}
data['term_class'] = data['term'].map(map_term).fillna('Unknown')
data = data.drop('term',axis = 1)

In [8]:
data['emp_length'].unique()

<StringArray>
['10+ years',   '3 years',   '4 years',   '6 years',    '1 year',   '7 years',
   '8 years',   '5 years',   '2 years',   '9 years',  '< 1 year',         nan]
Length: 12, dtype: str

In [9]:
map_emp_length = {
    '< 1 year' : 'Junior',
    '1 year' : 'Junior',
    '2 years' : 'Junior',
    '3 years' : 'Intermediate',
    '4 years' : 'Intermediate',
    '5 years' : 'Intermediate',
    '6 years' : 'Experienced',
    '7 years' : 'Experienced',
    '8 years' : 'Experienced',
    '9 years' : 'Experienced',
    '10 years' : 'Veteran',
    '10+ years' : 'Veteran',
}

data['emp_class'] = data['emp_length'].map(map_emp_length)
data = data.drop('emp_length',axis = 1)

In [10]:
default_map = {
    'Fully Paid': 'Paid',
    'Does not meet the credit policy. Status:Fully Paid': 'Paid',
    'Charged Off': 'Not Paid',
    'Default': 'Not Paid',
    'Does not meet the credit policy. Status:Charged Off': 'Not Paid'
}

data['default'] = data['loan_status'].map(default_map)
data = data.dropna(subset=['default'])

In [11]:
data = data.drop('loan_status',axis = 1)

In [12]:
data.sample(5)

,loan_amnt,int_rate,installment,grade,home_ownership,annual_inc,verification_status,purpose,dti,term_class,emp_class,default
2257861,40000.0,16.99,1425.92,D,RENT,96000.0,Source Verified,debt_consolidation,23.30,Short-term,Experienced,Paid
520983,14350.0,21.45,543.96,D,MORTGAGE,30205.5,Verified,debt_consolidation,47.68,Short-term,Intermediate,Paid
1148821,25325.0,12.99,853.18,C,MORTGAGE,196000.0,Verified,debt_consolidation,26.58,Short-term,Veteran,Paid
78968,20000.0,8.18,628.39,B,MORTGAGE,80000.0,Not Verified,debt_consolidation,11.06,Short-term,Intermediate,Paid
358168,35000.0,12.69,790.82,C,MORTGAGE,109000.0,Verified,debt_consolidation,21.28,Long-term,Experienced,Paid


In [13]:
print("Output class:",data['default'].value_counts())

Output class: default
Paid        1078739
Not Paid     269360
Name: count, dtype: int64


In [14]:
data_cleaned = data.copy()

In [15]:
data.isnull().sum()

loan_amnt                  0
int_rate                   0
installment                0
grade                      0
home_ownership             0
annual_inc                 4
verification_status        0
purpose                    0
dti                      374
term_class                 0
emp_class              78550
default                    0
dtype: int64

In [16]:
data['home_ownership'].unique()

<StringArray>
['MORTGAGE', 'RENT', 'OWN', 'ANY', 'NONE', 'OTHER']
Length: 6, dtype: str

In [17]:
data['verification_status'].unique()

<StringArray>
['Not Verified', 'Source Verified', 'Verified']
Length: 3, dtype: str

In [18]:
data['purpose'].unique()

<StringArray>
['debt_consolidation',     'small_business',   'home_improvement',
     'major_purchase',        'credit_card',              'other',
              'house',           'vacation',                'car',
            'medical',             'moving',   'renewable_energy',
            'wedding',        'educational']
Length: 14, dtype: str

In [19]:
data_cleaned.info()

<class 'pandas.DataFrame'>
Index: 1348099 entries, 0 to 2260697
Data columns (total 12 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   loan_amnt            1348099 non-null  float64
 1   int_rate             1348099 non-null  float64
 2   installment          1348099 non-null  float64
 3   grade                1348099 non-null  str    
 4   home_ownership       1348099 non-null  str    
 5   annual_inc           1348095 non-null  float64
 6   verification_status  1348099 non-null  str    
 7   purpose              1348099 non-null  str    
 8   dti                  1347725 non-null  float64
 9   term_class           1348099 non-null  str    
 10  emp_class            1269549 non-null  str    
 11  default              1348099 non-null  str    
dtypes: float64(5), str(7)
memory usage: 133.7 MB


In [20]:
data_cleaned.sample(5)

,loan_amnt,int_rate,installment,grade,home_ownership,annual_inc,verification_status,purpose,dti,term_class,emp_class,default
262671,12000.0,6.24,366.37,A,RENT,70000.0,Not Verified,credit_card,29.11,Short-term,Intermediate,Paid
197657,6000.0,11.53,197.95,B,MORTGAGE,39000.0,Not Verified,debt_consolidation,4.83,Short-term,Veteran,Paid
389178,12000.0,17.86,432.99,D,MORTGAGE,70000.0,Source Verified,other,17.20,Short-term,Experienced,Paid
54995,3400.0,13.67,115.66,C,MORTGAGE,19000.0,Verified,debt_consolidation,24.19,Short-term,NaN,Paid
2245313,28000.0,24.99,821.68,E,MORTGAGE,93000.0,Verified,debt_consolidation,30.79,Long-term,Experienced,Paid


In [21]:
print(data.isna().sum())

loan_amnt                  0
int_rate                   0
installment                0
grade                      0
home_ownership             0
annual_inc                 4
verification_status        0
purpose                    0
dti                      374
term_class                 0
emp_class              78550
default                    0
dtype: int64


DATA CLEANING AND TRANSFORMING PIPELINE

In [22]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('power', PowerTransformer(method='yeo-johnson')),
    ('scaler', StandardScaler())
])

In [23]:
log_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(np.log1p)),
    ('scaler', StandardScaler())
])

In [24]:
emp_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(
        categories=[['Junior','Intermediate','Experienced','Veteran']]
    ))
])

In [25]:
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [26]:
preprocessor = ColumnTransformer([
    ('log', log_pipeline, ['annual_inc','int_rate']),
    ('installment', numeric_pipeline, ['installment']),
    ('dti', numeric_pipeline, ['dti']),
    ('power', numeric_pipeline, ['loan_amnt']),
    ('emp', emp_pipeline, ['emp_class']),
    ('term',
     OrdinalEncoder(categories=[['Short-term','Long-term']]),
     ['term_class']),
    ('grade',
     OrdinalEncoder(categories=[['A','B','C','D','E','F','G']]),
     ['grade']),
    ('home', cat_pipeline, ['home_ownership']),
    ('verification', cat_pipeline, ['verification_status']),
    ('purpose', cat_pipeline, ['purpose'])
], remainder='passthrough')

In [27]:
x = data_cleaned.drop(['default'],axis=1)
y = data_cleaned['default']

In [28]:
x_train,x_test,y_train,y_test = train_test_split(x,y,
                                                 test_size = 0.2,
                                                 random_state = 42)

LOGISTIC REGRESSION

In [29]:
pipe_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(class_weight='balanced',
        max_iter=1000,
        random_state=42))
])

pipe_lr.fit(x_train, y_train)

c:\Users\Mayur23\OneDrive\Desktop\pareet\Loan Default Prediction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('log',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('log',
                                                                   FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['annual_inc', 'int_rate']),
                                                 ('installment',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('power',
                                                                   Pow...
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['verification_status']),
                                                 ('purpose',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['purpose'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

In [30]:
y_pred_lr = pipe_lr.predict(x_test)

In [31]:
accuracy_score(y_pred_lr,y_test)

0.6468437059565314

In [32]:
print(pd.Series(y_pred_lr).value_counts())

Paid        158402
Not Paid    111218
Name: count, dtype: int64


GRADIENTBOOSTINGCLASSIFIER

In [ ]:
pipe_gbc = Pipeline([
     ('preprocessor', preprocessor),
     ('model', GradientBoostingClassifier(n_estimators = 200))
 ])
pipe_gbc.fit(x_train, y_train)

In [ ]:
y_pred_gbc = pipe_gbc.predict(x_test)

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred_gbc))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_gbc))

In [ ]:
print(pd.Series(y_pred_gbc).value_counts())

In [ ]:
accuracy_score(y_pred_gbc,y_test)

In [ ]:
pipe_rf = Pipeline([
     ('preprocessor', preprocessor),
     ('model', RandomForestClassifier(
        n_estimators=200,
       class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

pipe_rf.fit(x_train, y_train)

y_pred_rf = pipe_rf.predict(x_test)

print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
accuracy_score(y_pred_rf,y_test)

In [40]:
from sklearn.metrics import classification_report, confusion_matrix

print("Classes:", pipe_lr.classes_)

not_paid_index = list(pipe_lr.classes_).index("Not Paid")

y_prob = pipe_lr.predict_proba(x_test)[:, not_paid_index]

for threshold in [0.50, 0.40, 0.30, 0.25, 0.20]:

    y_pred_threshold = [
        "Not Paid" if p >= threshold else "Paid"
        for p in y_prob
    ]

    print(f"\n Threshold: {threshold} ")
    print(classification_report(y_test, y_pred_threshold))
    

Classes: ['Not Paid' 'Paid']

 Threshold: 0.5 
              precision    recall  f1-score   support

    Not Paid       0.32      0.65      0.42     54136
        Paid       0.88      0.65      0.75    215484

    accuracy                           0.65    269620
   macro avg       0.60      0.65      0.58    269620
weighted avg       0.77      0.65      0.68    269620


 Threshold: 0.4 
              precision    recall  f1-score   support

    Not Paid       0.27      0.82      0.41     54136
        Paid       0.91      0.44      0.59    215484

    accuracy                           0.52    269620
   macro avg       0.59      0.63      0.50    269620
weighted avg       0.78      0.52      0.55    269620


 Threshold: 0.3 
              precision    recall  f1-score   support

    Not Paid       0.24      0.94      0.38     54136
        Paid       0.94      0.24      0.38    215484

    accuracy                           0.38    269620
   macro avg       0.59      0.59      0.38  

In [ ]:
# import pickle

# with open("model.pkl", "wb") as f:
#     pickle.dump(pipe_lr, f)

# print("Logistic regression pipeline saved!")

Logistic regression pipeline saved!


In [44]:
import os

size_mb = os.path.getsize("model.pkl") / (1024 ** 2)

print(f"Model size: {size_mb:.2f} MB")

Model size: 0.01 MB
